## Part 4 -- Runnable & LCEL
   ## Task 8: Runnable Basics

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from dotenv import load_dotenv
import re

load_dotenv()


model = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
) 

parser = StrOutputParser()


## PromptTemplate

## Answer prompt Template
answer_prompt = PromptTemplate(
                template = 'Answer this question im max 500 words: {question}', 
                input_variables = ['question'])

## Summary Prompt
summary_prompt = PromptTemplate(
                template = 'Give a short summary about this in 100 words: {answer}', 
                input_variables = ['answer'])  

## FollowUp Prompt
followup_prompt = PromptTemplate(
                template = 'Genearate 3 follow up question about: {answer}', 
                input_variables = ['answer'])  

 ## Creating Individual Chains for Each

answer_chain =   answer_prompt | model | parser
summary_chain = summary_prompt | model | parser
followup_chain = followup_prompt | model | parser


 ## Parallel Chain
parallel_chain = RunnableParallel({
    "answer": RunnablePassthrough(),
    "summary": summary_prompt | model | parser,
    "follow_up_questions": followup_prompt | model | parser 
    })


## Final Chain
final_chain = answer_chain | parallel_chain

  ## Invoke
response =  final_chain.invoke({"question": "How langchain works"})


# print(output['answer'])
# print(output['summary'])

print("Answer:", response["answer"])
print("Summary:", response["summary"])
print("Follow-up Questions:", response["follow_up_questions"])


Follow-up Questions: Here are three follow‑up questions you could ask after reading the “How LangChain Works” overview:

1. **How can I customize the memory component to retain only the most relevant parts of a long conversation, and what trade‑offs should I consider between full chat history vs. summarized buffers?**

2. **What are the best practices for designing a robust agent‑tool interaction loop (e.g., handling tool‑call failures, preventing infinite loops, and managing token usage) when building autonomous agents with LangChain?**

3. **If I need to replace the default `VectorDBRetriever` with a proprietary knowledge base (e.g., a SQL‑based document store), what steps are required to implement a custom retriever that still integrates seamlessly with LangChain’s chaining and memory mechanisms?**


## Task 9:  LCEL-Based RAG Chain

In [9]:

from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document
from langchain_core.runnables import RunnablePassthrough
from dotenv import load_dotenv

load_dotenv()

model = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)

parser = StrOutputParser()


# Documents
documents = [
    Document(page_content="LangChain is a framework for building applications using large language models."),
    Document(page_content="RAG stands for Retrieval Augmented Generation."),
    Document(page_content="LCEL stands for LangChain Expression Language."),
    Document(page_content="RunnablePassthrough passes input data without changing it.")
]


# Retriever function
def retriever(question):
    matching_documents = []

    for document in documents:
        if any(word in document.page_content.lower().split()
               for word in question.lower().split()):
            matching_documents.append(document.page_content)

    if matching_documents:
        return "\n".join(matching_documents)

    return "No relevant information found."


# Prompt Template
rag_prompt = PromptTemplate(
    template="""
Answer the question using the following context.
Do not use outside knowledge.

Context:
{context}

Question:
{question}

If the context does not contain the answer, say:
No relevant information found.
""",
    input_variables=["context", "question"]
)


# LCEL-Based RAG Chain
rag_chain = (
    RunnablePassthrough.assign(
        context=lambda x: retriever(x["question"])
    )
    | rag_prompt
    | model
    | parser
)


# Test with multiple user queries
questions = [
    "What is LangChain?",
    "What does RAG stand for?",
    "What is LCEL?",
    "What does RunnablePassthrough do?",
    "What is Python?"
]

for question in questions:
    response = rag_chain.invoke({"question": question})

    print("\nQuestion:", question)
    print("Answer:", response)
    print("-" * 50)



Question: What is LangChain?
Answer: LangChain is a framework for building applications using large language models.
--------------------------------------------------

Question: What does RAG stand for?
Answer: RAG stands for **Retrieval Augmented Generation**.
--------------------------------------------------

Question: What is LCEL?
Answer: No relevant information found.
--------------------------------------------------

Question: What does RunnablePassthrough do?
Answer: RunnablePassthrough simply passes the input data through unchanged.
--------------------------------------------------

Question: What is Python?
Answer: No relevant information found.
--------------------------------------------------


## Task 10: Observations and Insights
## Q.1 Why structured output is important
## Ans: Structured output organizes LLM responses into a predefined format, such as JSON or a Pydantic model. It makes responses easier to parse, validate, store in databases, and use in applications.

## Q.2: Advantages of LCEL over traditional chains
## Ans: 
## 1) Simple syntax: Connect components using the | operator.
## 2) Reusability: Build and reuse individual chains easily.
## 3) Composition: Combine sequential, parallel, and conditional workflows.
## 4) Streaming support: Stream outputs when supported by the components.
## 5) Integration: Works with prompts, models, retrievers, and output parsers


## Q.3: When to use parallel vs. conditional chains?
## Ans: Parallel chains: Use when multiple independent tasks need to run on the same input simultaneously. For example, generating an answer, summary, and follow-up questions.

## Conditional chains: Use when the next step depends on a condition. For example, using retrieval when relevant documents exist and a different chain when they do not.